# Download OpenForensics dataset

Downloads the [OpenForensics](https://zenodo.org/records/5528418) dataset (Le et al., ICCV 2021, CC BY 4.0) from Zenodo.

Made for **Google Colab**. Files are extracted to Colab's local disk (`/content`), which is fast but **erased when the runtime disconnects**, so you'll re-run this notebook each session.

The full dataset is about **67 GB** of zips, which won't fit on Colab's disk, so choose the parts you need in `FILES_TO_GET`. Downloads resume if they're interrupted, are checked against Zenodo's MD5, and are then extracted.

**Optional:** set `USE_DRIVE = True` to keep the zips in Google Drive so later sessions skip the download and only re-extract. This needs enough Drive space (the starter subset is about 13 GB, and free Drive is 15 GB).

In [ ]:
import hashlib
import json
import shutil
import urllib.request
import zipfile
from pathlib import Path

from tqdm.auto import tqdm

RECORD_ID = "5528418"
USE_DRIVE = False  # True = keep zips in Google Drive between sessions

DATA_DIR = Path("/content/data/OpenForensics")  # extracted images (local, fast)
if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    ZIP_DIR = Path("/content/drive/MyDrive/OpenForensics_zips")
else:
    ZIP_DIR = Path("/content/zips")

# Local zips are deleted after extraction to free disk; Drive zips are kept
DELETE_ZIPS_AFTER_EXTRACT = not USE_DRIVE

DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_DIR.mkdir(parents=True, exist_ok=True)
print(f"Free space on Colab disk: {shutil.disk_usage('/content').free / 1e9:.0f} GB")

## List the files in the Zenodo record

In [ ]:
with urllib.request.urlopen(f"https://zenodo.org/api/records/{RECORD_ID}") as r:
    record = json.load(r)

files = {f["key"]: f for f in record["files"]}
for name in sorted(files):
    print(f"{name:30s} {files[name]['size'] / 1e9:6.2f} GB")
print(f"Total: {sum(f['size'] for f in files.values()) / 1e9:.1f} GB")

## Choose what to download

- `Train_part_1..5.zip`: training images (~20 GB total)
- `Val.zip`: validation images (~3 GB)
- `Test-Dev_part_1..2.zip`: test images (~8 GB)
- `Test-Challenge_part_1..5.zip`: harder test set with extra perturbations (~23 GB)
- `*_poly.json`: COCO-style annotations (real/fake label, box and mask for each face)

The default below is a starter subset of about 12 GB.

In [ ]:
FILES_TO_GET = [
    "readme.txt",
    "Train_part_1.zip",
    "Train_poly.json",
    "Val.zip",
    "Val_poly.json",
    "Test-Dev_part_1.zip",
    "Test-Dev_poly.json",
]

# Uncomment to download everything:
# FILES_TO_GET = sorted(files)

total = sum(files[n]["size"] for n in FILES_TO_GET)
largest_zip = max((files[n]["size"] for n in FILES_TO_GET if n.endswith(".zip")), default=0)
# Zips are extracted and deleted one at a time, so peak use is all extracted data plus one zip
needed = total + (0 if USE_DRIVE else largest_zip)
free = shutil.disk_usage("/content").free
print(f"Selected {len(FILES_TO_GET)} files, {total / 1e9:.1f} GB")
print(f"Need about {needed / 1e9:.1f} GB on Colab disk, {free / 1e9:.0f} GB free")
if needed > free:
    print("WARNING: not enough space, remove some files from FILES_TO_GET")

## Download, verify and extract

Each zip is downloaded, MD5-checked and extracted before the next one starts, so the disk never holds more than one zip at a time.

In [ ]:
def md5sum(path, chunk=1 << 20):
    h = hashlib.md5()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(chunk), b""):
            h.update(block)
    return h.hexdigest()


def download(info, dest):
    size = info["size"]
    url = info["links"]["self"]
    have = dest.stat().st_size if dest.exists() else 0
    if have == size:
        return
    if have > size:
        dest.unlink()
        have = 0

    req = urllib.request.Request(url, headers={"Range": f"bytes={have}-"} if have else {})
    with urllib.request.urlopen(req) as r:
        if have and r.status != 206:  # server ignored the Range header, start over
            have = 0
        with open(dest, "ab" if have else "wb") as f, tqdm(
            total=size, initial=have, unit="B", unit_scale=True, desc=dest.name
        ) as bar:
            while chunk := r.read(1 << 20):
                f.write(chunk)
                bar.update(len(chunk))


for name in FILES_TO_GET:
    info = files[name]
    dest = (ZIP_DIR if name.endswith(".zip") else DATA_DIR) / name
    extracted_marker = DATA_DIR / f".{name}.done"
    if extracted_marker.exists():
        print(f"{name}: already downloaded and extracted")
        continue

    for attempt in range(3):
        download(info, dest)
        if md5sum(dest) == info["checksum"].split(":", 1)[1]:
            print(f"{name}: OK")
            break
        print(f"{name}: checksum mismatch, re-downloading")
        dest.unlink()
    else:
        raise RuntimeError(f"{name} failed checksum 3 times")

    if name.endswith(".zip"):
        with zipfile.ZipFile(dest) as z:
            for member in tqdm(z.infolist(), desc=f"Extracting {name}"):
                z.extract(member, DATA_DIR)
        if DELETE_ZIPS_AFTER_EXTRACT:
            dest.unlink()
    extracted_marker.touch()

print("Contents of", DATA_DIR.resolve())
for p in sorted(DATA_DIR.iterdir()):
    if not p.name.startswith((".", "_")):
        print(" ", p.name)

## Quick look at the annotations

In [ ]:
with open(DATA_DIR / "Val_poly.json") as f:
    val = json.load(f)

print("Keys:", list(val.keys()))
print("Images:", len(val["images"]), "| Faces:", len(val["annotations"]))
print("Categories:", val["categories"])